In [ ]:
# imports

import pandas as pd
import sqlite3
from os.path import isfile, join
import os
from os import listdir
import json
import shutil
from math import ceil
import csv
import posixpath
import paramiko
from tqdm.auto import tqdm
import numpy as np

In [ ]:
# functions needed

def get_urls(ref_list):
    urls = []
    try: 
        if isinstance(ref_list,str):
            for ref in json.loads(ref_list.replace("'", '"')):
                urls.append(ref['url'])
        elif isinstance(ref_list, list):
            for element in ref_list:
                for ref in json.loads(element):
                    urls.append(ref['url'])
    except:
        pass
    return urls

def get_files(df, path_folder, path_ds):
    for index, row in df.iterrows():
        cwe = row['cwe_id'].replace('-','_')
        file_name_vul = row['safe_filename_vulcode']
        root_vul = join(path_ds, row['file_path_vulcode'])
        file_name_patch = row['safe_filename_patch']
        root_patch = join(path_ds, row['file_path_patch'])
        try: 
            os.makedirs(join(path_folder, 'code_before', cwe), exist_ok=True)
            shutil.copyfile(root_vul, join(path_folder, 'code_before', cwe, file_name_vul))
            os.makedirs(join(path_folder, 'code_patch', cwe), exist_ok=True)
            shutil.copyfile(root_patch, join(path_folder, 'code_patch', cwe, file_name_patch))
        except Exception as e:
            print(e)

### create the dataset for the evaluation

In [ ]:
path_cwe_cve = r".\\metadata\\cwe_classification.csv"
cve_cwe_match_df = pd.read_csv(path_cwe_cve)
print(f"Length: {len(cve_cwe_match_df.index)}")

In [ ]:
# get the cve with the urls

path_metadata_cve_urls = r".\\metadata\\cve_data.csv"
meta_cve_url = pd.read_csv(path_metadata_cve_urls, usecols=['cve_id', 'reference_json'])
meta_cve_url['repo_url'] = meta_cve_url['reference_json'].apply(get_urls)
print(f"Number of found CVEs: {len(meta_cve_url.index)}")

meta_cve_url_big = meta_cve_url.explode('repo_url')
print(f"Number of found URLs for the CVEs: {len(meta_cve_url_big.index)}")

meta_cve_url_big = meta_cve_url_big.dropna()
clean_meta_cve_url_big = meta_cve_url_big.loc[meta_cve_url_big['repo_url'].str.contains('github')]
clean_meta_cve_url_big = clean_meta_cve_url_big.drop_duplicates()
print(f"Number of clean URLs for the CVEs (contains github link): {len(clean_meta_cve_url_big.index)}")

In [ ]:
# get all filenames of the vulnerable code metadata 

root = r".\\metadata\\vul_metadata"

file_names = []
for path, subdirs, files in os.walk(root):
    for name in files:
        file_names.append(os.path.join(path, name))
df_filename = pd.concat((pd.read_csv(file) for file in file_names), ignore_index=True)
df_filename = df_filename.drop_duplicates()
print(f"Number of filenames: {len(df_filename.index)}")

In [ ]:
# merge the filenames with cve ids

df_merge = pd.merge(clean_meta_cve_url_big, df_filename, on=['repo_url'])
df_merge_cwe = pd.merge(df_merge, cve_cwe_match_df, on=['cve_id'])
df_merge_cwe = df_merge_cwe.drop_duplicates()

print(f"Number of matches (filenames, cwe): {len(df_merge_cwe.index)}")

In [ ]:
# clean non informative CWE taggs

df_clean_cwe = df_merge_cwe[~df_merge_cwe["cwe_id"].apply(lambda x: x in ["NVD-CWE-noinfo", "NVD-CWE-Other"])]
print(f"Number of samples with CWE: {len(df_clean_cwe.index)}")

In [ ]:
# get file endings

df_clean_cwe["file_ending"] = df_clean_cwe["original_filename"].apply(lambda x: x.split(".")[-1])

In [ ]:
# final cleaning of the languages

final_df = df_clean_cwe[df_clean_cwe["original_filename"].apply(lambda x: x.split(".")[-1] in ["py", "java", "c", "cpp"])]
final_df = final_df.reset_index(drop=True)
final_df = final_df.drop_duplicates()
print(f"Number of clean samples: {len(final_df.index)}")

In [ ]:
# get filename dataframe of patches

file_names = []

for path, subdirs, files in os.walk(r".\\metadata\\patch_metadata"):
    for name in files:
        file_names.append(os.path.join(path, name))


df_filename_patches = pd.concat((pd.read_csv(f) for f in file_names), ignore_index=True)
df_filename_patches = df_filename_patches.drop_duplicates()
print(f"Number of patches: {len(df_filename.index)}")

In [ ]:
# left join of vul code and patches

df_vul_patches_final= final_df.merge(df_filename_patches, how='left', on=['repo_url', 'commit_sha'])
df_vul_patches_final = df_vul_patches_final.drop_duplicates()
df_vul_patches_final = df_vul_patches_final.rename(columns={"safe_filename_y": "safe_filename_patch", "file_path_y": "file_path_patch",
                                                            "safe_filename_x": "safe_filename_vulcode", "file_path_x": "file_path_vulcode"})

print(f"Number of rows (contains nan values): {len(df_vul_patches_final.index)}")

In [ ]:
# clean nan values
df_vul_patches_final_clean = df_vul_patches_final.drop_duplicates()
df_vul_patches_final_clean = df_vul_patches_final_clean.dropna()
print(f"Number of rows (final cleaned): {len(df_vul_patches_final_clean.index)}")

In [ ]:
print("Number of code samples:")
print(f"Number of matches (filenames, cwe): {len(df_merge_cwe.index)}")
print(f"Number of files with valid cwe label: {len(df_clean_cwe.index)}")
print(f"Number of files with valid cwe label and correct languages: {len(final_df.index)}")
print(f"Number of rows patch and vul code matched (could contain nan values): {len(df_vul_patches_final.index)}")
print(f"Number of rows patch and vul code matched (without nan values): {len(df_vul_patches_final_clean.index)}")

In [ ]:
print("Number of urls:")
print(f"Number of matches (filenames, cwe): {df_merge_cwe["repo_url"].nunique()}")
print(f"Number of urls with valid cwe label: {df_clean_cwe["repo_url"].nunique()}")
print(f"Number of urls with valid cwe label and correct languages: {final_df["repo_url"].nunique()}")
print(f"Number of urls patch and vul code matched (could contain nan values): {df_vul_patches_final["repo_url"].nunique()}")
print(f"Number of urls patch and vul code matched (without nan values): {df_vul_patches_final_clean["repo_url"].nunique()}")

### reorganizing the code files on the server for the SAST Tool

In [ ]:
def organize_on_server(rows, batch_size=1000, base_dir='data_final_batches'):
    num_batches = ceil(len(rows) / batch_size)

    for batch_num in range(num_batches):
        batch_start = batch_num * batch_size
        batch_end = min((batch_num + 1) * batch_size, len(rows))
        batch_rows = rows[batch_start:batch_end]

        batch_folder = os.path.join(base_dir, f'batch_{batch_num + 1}')
        vul_code_folder = os.path.join(batch_folder, 'code_before')
        patch_folder = os.path.join

        os.makedirs(vul_code_folder, exist_ok=True)
        os.makedirs(patch_folder, exist_ok=True)

        for row in batch_rows:
            cwe = row['cwe_id'].replace('-', '_')
            vul_cwe_folder = os.path.join('code_before', cwe)
            patch_cwe_folder = os.path.join('code_after', cwe)
            os.makedirs(vul_cwe_folder, exist_ok=True)
            os.makedirs(patch_cwe_folder, exist_ok=True)

            vul_path = row['file_path_vulcode'].replace('\\', '/')
            patch_path = row['file_path_patch'].replace('\\', '/')

            vul_filename = os.path.basename(vul_path)
            patch_filename = os.path.basename(patch_path)

            vul_target = os.path.join(vul_cwe_folder, vul_filename)
            patch_target = os.path.join(patch_cwe_folder, patch_filename)

            try:
                shutil.copy2(os.path.join("/data/users/PUBLIC/vulnerability-dataset/", vul_path), vul_target)
            except Exception as e:
                print(f"Error copying from {vul_path}: {e}")
            try:
                shutil.copy2(os.path.join("/data/users/PUBLIC/vulnerability-dataset/", patch_path), patch_target)
            except Exception as e:
                print(f"Error copying from {patch_path}: {e}")

    print(f"Files were copied and organized into {num_batches} batches.")

In [ ]:
current_dir = os.path.dirname(os.path.abspath(__file__))

csv_path = os.path.join(current_dir, r".\\metadata\\custom_ds_metadata_combined_cleaned_withoutNan.csv")
output_dir = os.path.join(current_dir, "data_final_batches")
os.makedirs(output_dir, exist_ok=True)

if not os.path.isfile(csv_path):
    print(f"csv-file not found: {csv_path}")
else:
    print(f"Load csv-file: {csv_path}")
    rows = read_csv(csv_path)
    organize_on_server(rows, batch_size=len(rows), base_dir=output_dir)

### merge with SAST results

In [ ]:
# create sample_name to merge with the verdicts

df_vul_patches_final_clean["cwe_path_id"] = df_vul_patches_final_clean["cwe_id"].str.replace('-','_')
df_vul_patches_final_clean["sample_name"] = df_vul_patches_final_clean["cwe_path_id"].astype(str).str.lower() + "_" + df_vul_patches_final_clean["safe_filename_vulcode"].astype(str).str.lower()

In [ ]:
# final merge with verdicts from semgrep

df_verdict = pd.read_csv(r".\\metadata\\scan_result_evaluation_sample_based.csv")
df_verdict_merged = df_vul_patches_final_clean.merge(df_verdict, on=["sample_name"], how="left")
df_verdict_merged.drop_duplicates()

### creating the jsonl file for llm training

In [ ]:
df_verdict_merged = df_verdict_merged.drop(columns=['finding_id', 'file_path', 'filename', 'status', 'cwe_id_clean',
                 'cwe_gt_clean', 'sample_name', 'children', 'parents', 'related',
                 'tp_count', 'fp_count',
                 'ep_count', 'reference_json',
                 'safe_filename_vulcode', 'safe_filename_patch',
                 'cwe_id', 'batch_nr', "final_verdict", 
                 'cwe_path_id', 'server_filename_vulcode'])

In [ ]:
df_verdict_merged = df_verdict_merged.rename(columns={"groundtruth_cwe_id": "cwe_original", 
                        "found_cwe_id": "cwe_detected", 
                        "match_result": "match_type", 
                        "file_ending": "language", 
                        "applied_semgrep_rule": "rules_triggered",
                        'original_filename':'file_path'
                       })

In [ ]:
def static_match(x):
    return True if x=='EP' else False

def cwe_final(x, d, o):
    return d if x in ['Mismatch', 'EP'] else o

In [ ]:
df_verdict_merged["timestamp"] = df_verdict_merged["file_path_vulcode"].str.split("\\").str[1]
df_verdict_merged["timestamp"]= df_verdict_merged["timestamp"].str.split("_").str[-1]
df_verdict_merged["timestamp"] = pd.to_datetime(df_verdict_merged["timestamp"],format='%d.%m.%Y')
df_verdict_merged["timestamp"] = df_verdict_merged["timestamp"].astype(str)
df_verdict_merged['match_type']= df_verdict_merged['match_type'].astype(str)
df_verdict_merged["static_match"] = df_verdict_mergeddf['match_type'].apply(static_match)
df_verdict_merged['match_type'] = df_verdict_merged['match_type'].str.replace('FP','Mismatch')
df_verdict_merged['match_type'] = df_verdict_merged['match_type'].str.replace('TP','HP')
df_verdict_merged['cwe_final'] = df_verdict_merged.apply(lambda x: cwe_final(x['match_type'], x['cwe_detected'], x['cwe_original']), axis = 1)

In [ ]:
df_vul_code = df['file_path_vulcode'].drop_duplicates().to_frame()
df_patch_code = df['file_path_patch'].drop_duplicates().to_frame()
vul_list = []
patch_list = []


for i, row in tqdm(df_vul_code.iterrows(), total=len(df_vul_code), position=0, leave=True):
    file_path = row['file_path_vulcode'].replace('\\', '/')
    code_before = ""
    try:
        file_path = row['file_path_vulcode'].replace('\\', '/')
        code_before = ""
        with file(file_path, "rb") as f:
            for line in f:
                code_before += line.decode("utf-8")
    except:
        code_before = None
    vul_list.append(code_before)

se_vul = pd.Series(vul_list)
df_vul_code['code'] = se_vul.values

for i, row in tqdm(df_patch_code.iterrows(), total=len(df_patch_code), position=0, leave=True):
    try:
        file_path = row['file_path_patch'].replace('\\', '/')
        code_after = ""
        with file(file_path "rb") as f:
            for line in f:
                code_after += line.decode("utf-8")
    except:
        code_after = None
    patch_list.append(code_after)

se_patch = pd.Series(patch_list)
df_patch_code['code'] = se_patch.values

In [ ]:
df_merged_vul = df.merge(df_vul_code, on=["file_path_vulcode"], how="left")
df_merged_vul = df_merged_vul.rename(columns={"code": "code_before"})
df_merged_final = df_merged_vul.merge(df_patch_code, on=["file_path_patch"], how="left")
df_merged_final = df_merged_final.rename(columns={"code": "code_after"})

In [ ]:
# create jsonl file
with open("validated.jsonl", "w") as outfile:
    for i, row in tqdm(df_merged_final.iterrows(), total=len(df_merged_final), position=0, leave=True):   
        dict_row = row.to_dict()
        out = json.dumps(dict_row)
        outfile.write(out + "\n")

In [ ]:
with open("raw.jsonl", "w") as outfile:
    for i, row in tqdm(df_vul_patches_final_clean.iterrows(), total=len(df_vul_patches_final_clean), position=0, leave=True):
    
        # code before
        try:
            file_path = row['file_path_vulcode'].replace('\\', '/')
            code_before = ""
            with file(file_path, "rb") as f:
                for line in f:
                    code_before += line.decode("utf-8")
        except:
            code_before = None
    
        # code after
        try:
            file_path = row['file_path_patch'].replace('\\', '/')
            code_after = ""
            with file(file_path "rb") as f:
                for line in f:
                    code_after += line.decode("utf-8")
        except:
            code_after = None
            
        dict_row = row.to_dict()
        dict_row["code_before"] = code_before
        dict_row["code_after"] = code_after
        out = json.dumps(dict_row)
        outfile.write(out + "\n")